# Muertes por coronavirus
Análisis de datos de un dataset con reportes de muertes por coronavirus en el 2020-2021

Usaremos indexación, asignación y reordenamiento de columnas para dividir dos datasets con información de vacunación y muertes por COVID-19, esto con el objetivo de importar los dos archivos en una base de datos internas y hacer un análisis por medio de sentencias SQL

# Librerias

Estas serán las únicas librerias que usaremos, pandas para lectura de nuestro excel principal, también para creación y manipulación de dataframes. La libreria os será para manejo de rutas relativas y lectura de carpetas

In [1]:
import pandas as pd
import os

# Directorio

Creamos la carpeta de nuestros datasets si no existe

In [2]:
path = os.getcwd()
directorio = os.path.join(path, "datasets/")
if not os.path.exists(directorio):
    os.makedirs(directorio)

## Datasets

Debemos mover nuestros datasets a la carpeta que creamos, esto recorre e inserta dentro de una lista todos los archivos de excel o csv que puedan estar dentro de este directorio.

In [ ]:
path, dirs, files = next(os.walk(directorio))
file_count = len(files)
df_list = []

for i in range(file_count):
    if files[i].endswith(".xlsx"):
        temp_df = pd.read_excel(directorio + files[i], engine='openpyxl')
    elif files[i].endswith(".csv"):
        temp_df = pd.read_csv(directorio + files[i])
    else:
        print(f"Archivo no soportado: {files[i]}") # Si el archivo no es ni .xlsx ni .csv, saltá de inmediato un error, evitando que el ciclo siga ejecutandose, pero manteniendo los archivos válidos en la lista.
        continue
    df_list.append(temp_df)

# Operaciones

## Covid Deaths Dataset

Hay que comenzar por definir el dataset con el cual trabajaremos, en mi caso es el primer elemento de la lista, lo convertimos en un dataframe.

In [9]:
df_covid = pd.DataFrame(df_list[0])

In [ ]:
df_covid.head(2) # Preview de nuestro dataframe

In [ ]:
df_covid.columns # Esto me facilita mucho el tema de reordenarmiento de columnas, renombrado, etc. 

Dataset ordenado

Simplemente copio el output obtenido y cambio el orden de mis columnas, en este caso quiero poner "population" al frente de las columnas con las que más trabajaré, y para evitar tener que hacer un join on cada vez que quiera trabajar con esta columna.

In [ ]:
df_covid = df_covid.loc[:, ['iso_code', 'continent', 'location', 'date', 'population', 'total_cases', 'new_cases', # Aquí
       'new_cases_smoothed', 'total_deaths', 'new_deaths',
       'new_deaths_smoothed', 'total_cases_per_million',
       'new_cases_per_million', 'new_cases_smoothed_per_million',
       'total_deaths_per_million', 'new_deaths_per_million',
       'new_deaths_smoothed_per_million', 'reproduction_rate', 'icu_patients',
       'icu_patients_per_million', 'hosp_patients',
       'hosp_patients_per_million', 'weekly_icu_admissions',
       'weekly_icu_admissions_per_million', 'weekly_hosp_admissions',
       'weekly_hosp_admissions_per_million', 'new_tests', 'total_tests',
       'total_tests_per_thousand', 'new_tests_per_thousand',
       'new_tests_smoothed', 'new_tests_smoothed_per_thousand',
       'positive_rate', 'tests_per_case', 'tests_units', 'total_vaccinations',
       'people_vaccinated', 'people_fully_vaccinated', 'new_vaccinations',
       'new_vaccinations_smoothed', 'total_vaccinations_per_hundred',
       'people_vaccinated_per_hundred', 'people_fully_vaccinated_per_hundred',
       'new_vaccinations_smoothed_per_million', 'stringency_index', 'population_density', 'median_age', 'aged_65_older',
       'aged_70_older', 'gdp_per_capita', 'extreme_poverty',
       'cardiovasc_death_rate', 'diabetes_prevalence', 'female_smokers',
       'male_smokers', 'handwashing_facilities', 'hospital_beds_per_thousand',
       'life_expectancy', 'human_development_index']]

Dataset para operar con datos de muertes por covid

Ahora, estas son las columnas para analizar muertes por covid, por fecha, población, país o locación y continente, con esto puedo hacer un análisis completo asegurando una mayor comodidad a la hora de crear sentencias SQL.

In [13]:
df_covidDeaths = df_covid.loc[:,['iso_code', 'continent', 'location', 'date', 'population', 'total_cases', 'new_cases',
       'new_cases_smoothed', 'total_deaths', 'new_deaths',
       'new_deaths_smoothed', 'total_cases_per_million',
       'new_cases_per_million', 'new_cases_smoothed_per_million',
       'total_deaths_per_million', 'new_deaths_per_million',
       'new_deaths_smoothed_per_million', 'reproduction_rate', 'icu_patients',
       'icu_patients_per_million', 'hosp_patients',
       'hosp_patients_per_million', 'weekly_icu_admissions',
       'weekly_icu_admissions_per_million', 'weekly_hosp_admissions',
       'weekly_hosp_admissions_per_million']]

Dataset para operar con datos de vacunaciones

Este es nuestro dataframe con información acerca de las vacunaciones, podemos hacer el mismo análisis que con el dataframe de muertes, y podemos hacer un join para poder analizar las vacunaciones por población.

In [14]:
df_covidVac = df_covid.loc[:,['iso_code', 'continent', 'location', 'date', 'new_tests', 'total_tests',
       'total_tests_per_thousand', 'new_tests_per_thousand',
       'new_tests_smoothed', 'new_tests_smoothed_per_thousand',
       'positive_rate', 'tests_per_case', 'tests_units', 'total_vaccinations',
       'people_vaccinated', 'people_fully_vaccinated', 'new_vaccinations',
       'new_vaccinations_smoothed', 'total_vaccinations_per_hundred',
       'people_vaccinated_per_hundred', 'people_fully_vaccinated_per_hundred',
       'new_vaccinations_smoothed_per_million', 'stringency_index',
       'population', 'population_density', 'median_age', 'aged_65_older',
       'aged_70_older', 'gdp_per_capita', 'extreme_poverty',
       'cardiovasc_death_rate', 'diabetes_prevalence', 'female_smokers',
       'male_smokers', 'handwashing_facilities', 'hospital_beds_per_thousand',
       'life_expectancy', 'human_development_index']]

## Exportar todos nuestros datasets a excel

Recordatorio:

- Estos DataFrame son bastante pequeños, por ende solo usaremos el método tradicional para exportarlos a excel, cuando los dataframes sobrepasen las 100000 filas, deberiamos cargarlo por chunks de 50k

In [18]:
df_covid.to_excel(directorio + "CovidData.xlsx", index=False)
df_covidDeaths.to_excel(directorio + "CovidDeathsUpdated.xlsx", index=False)
df_covidVac.to_excel(directorio + "CovidVaccinations.xlsx", index=False)

Método con Chunks

In [ ]:
# chunk_size = 50000
# with pd.ExcelWriter(directorio +"CovidData.xlsx", engine="xlsxwriter") as writer:
#     startrow = 0
#     recorrido = 0
#     for i in range(0, len(df_covid), chunk_size):
#         chunk = df_covid.iloc[i:i+chunk_size]
#         chunk.to_excel(
#             writer,
#             sheet_name="data",
#             startrow=startrow,
#             index=False,
#             header=(i == 0)
#         )

#         startrow += len(chunk)
#         recorrido += 1
# print(f"Vueltas: {recorrido}")